In [ ]:
# -*- coding: utf-8 -*-
import re, numpy as np, pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression

# ===================== CONFIGURAÇÕES =====================
REF_TEMP      = 20
PKL_TREINO    = "base_treino.pkl"
PKL_PROVA     = "base_prova (1).pkl"
TEMPS_TREINO = {0, 10, 40, 60}
TEMPS_PROVA  = {-10, 30, 46, 70}
CAP_GAIN_FRAC   = 0.60
CAP_OFFSET_FRAC = 0.60
CAP_TILT_FRAC   = 0.40

bands = [(35,50),(35,75),(35,100),(35,125)]

# ===================== FUNÇÕES AUXILIARES =====================
def extract_freq_hz(col):
    m = re.match(r"^f_(\d+(?:\.\d+)?)Hz$", str(col))
    return float(m.group(1)) if m else None

def get_freq_columns(df, fmin_khz, fmax_khz):
    cols, freqs = [], []
    for c in df.columns:
        f = extract_freq_hz(c)
        if f is not None:
            fk = f/1e3
            if fmin_khz <= fk <= fmax_khz:
                cols.append(c); freqs.append(f)
    order = np.argsort(freqs)
    return [cols[i] for i in order], np.array(freqs, float)[order]

def energy_weighted_centroid(f, x):
    xm = np.asarray(x, float)
    w  = xm*xm
    den = float(np.trapezoid(w, f))
    if den <= 1e-18: return float(np.mean(f))
    num = float(np.trapezoid(f*w, f))
    return num/den

def slope_over_band(f, x):
    return float((x[-1]-x[0])/(f[-1]-f[0] + 1e-12))

def compute_features(X, f):
    X = np.asarray(X, float); n, m = X.shape
    out=[]
    cuts = np.linspace(f[0], f[-1], 6)
    for i in range(n):
        x = X[i]
        mean  = float(np.mean(x))
        std   = float(np.std(x))
        amp   = float(x.max() - x.min())
        slope = slope_over_band(f, x)
        pk_i  = int(np.argmax(x)); peak_pos_rel = pk_i / max(1,(m-1))
        centroid = energy_weighted_centroid(f, x)
        z = (x - mean)/(std + 1e-12)
        skew = float(np.mean(z**3))
        kurt = float(np.mean(z**4))
        E_bands=[]
        for j in range(len(cuts)-1):
            mask = (f>=cuts[j]) & (f<cuts[j+1])
            if mask.sum()<2: E_bands.append(0.0)
            else: E_bands.append(float(np.trapezoid((x[mask]**2), f[mask])))
        out.append([mean,std,amp,slope,peak_pos_rel,centroid,skew,kurt,*E_bands])
    cols = ["mean","std","amp","slope","peak_pos_rel","centroid","skew","kurt",
            "E_b1","E_b2","E_b3","E_b4","E_b5"]
    return np.array(out, float), cols

def fit_feature_vs_temp_models(F, T, names):
    models = {}
    T = np.asarray(T, float).reshape(-1,1)
    for j, name in enumerate(names):
        lr = LinearRegression().fit(T, F[:,j])
        models[name] = lr
    return models

def feature_targets_at_ref(models, ref_temp=REF_TEMP):
    Tref = np.array([[ref_temp]])
    return {name: float(lr.predict(Tref)[0]) for name,lr in models.items()}

def shift_interp(x_row, fhz, tau_hz):
    f_shift = fhz + float(tau_hz)
    return np.interp(fhz, f_shift, x_row, left=x_row[0], right=x_row[-1])

def apply_compensation_by_features(x, f, targets, caps, y_ref=None):
    x = x.copy()
    mean_t = targets["mean"]; amp_t  = targets["amp"]
    slope_t= targets["slope"]; centroid_t = targets.get("centroid", None)
    mean_x = float(x.mean()); amp_x  = float(x.max() - x.min())
    slope_x= slope_over_band(f, x)
    offset = mean_t - mean_x
    offset_cap = caps["offset_frac"] * max(1e-9, amp_x)
    offset = float(np.clip(offset, -offset_cap, offset_cap))
    x = x + offset
    gain = 1.0 if amp_x<=1e-9 else float(amp_t/amp_x)
    gmin = 1.0 - caps["gain_frac"]; gmax = 1.0 + caps["gain_frac"]
    gain = float(np.clip(gain, gmin, gmax))
    x = mean_t + gain*(x - mean_t)
    delta_slope = slope_t - slope_x
    u = np.linspace(-0.5, 0.5, len(x))
    df = (f[-1]-f[0] + 1e-12)
    tilt_signal = (delta_slope * df) * u
    tilt_cap = caps["tilt_frac"] * max(1e-9, amp_x)
    tilt_signal = np.clip(tilt_signal, -tilt_cap, tilt_cap)
    x = x + tilt_signal
    if centroid_t is not None:
        cent_x = energy_weighted_centroid(f, x)
        delta_c = centroid_t - cent_x
        tau_max = 0.025 * (f[-1]-f[0])
        tau = float(np.clip(delta_c, -tau_max, tau_max))
        if abs(tau) > 1e-12:
            x = shift_interp(x, f, tau)
    return x

def compensate_set_by_features(X, f, feat_models, ref_temp, y_ref, caps):
    targets = feature_targets_at_ref(feat_models, ref_temp)
    Y = np.zeros_like(X)
    for i in range(X.shape[0]):
        yi = apply_compensation_by_features(X[i], f, targets, caps, y_ref=y_ref)
        Y[i] = yi
    return Y, targets

def park_compensate_single(x, y_ref, fhz,
                           max_shift_frac=0.25,
                           overlap_min_frac=0.60):
    n = len(x); fmin, fmax = fhz[0], fhz[-1]; df_band = fmax - fmin
    tau_max = max_shift_frac * df_band
    nsteps = 101; tau_vals = np.linspace(-tau_max, tau_max, nsteps)
    best = (np.inf, 0.0, 0.0)
    for tau in tau_vals:
        x_shift = shift_interp(x, fhz, tau)
        deltaS = float(np.mean(y_ref - x_shift))
        resid = y_ref - (x_shift + deltaS)
        Va = float(np.sum(resid*resid))
        if Va < best[0]: best = (Va, tau, deltaS)
    _, tau_best, dS_best = best
    yout = shift_interp(x, fhz, tau_best) + dS_best
    return yout, tau_best, dS_best

def park_batch(X, y_ref, fhz):
    n, m = X.shape; Y = np.zeros_like(X)
    for i in range(n):
        yi, _, _ = park_compensate_single(X[i], y_ref, fhz)
        Y[i] = yi
    return Y, None, None

# ===================== FUNÇÃO DE MÉTRICAS =====================
def run_all_metrics(y_ref, X_te, Y_rf, Y_pk, fhz, rf_temp, feat_names):
    def ST_puro(y_ref, y_hat, f, rf_model, feat_names, eps=1e-9):
        feats_ref, _ = compute_features(y_ref[None,:], f)
        feats_hat, _ = compute_features(y_hat[None,:], f)
        sel = ["mean","amp","slope","centroid"]
        importances = dict(zip(feat_names, rf_model.feature_importances_))
        weights = np.array([importances.get(n,0) for n in sel], float)
        if np.sum(weights)<=0: weights = np.ones_like(weights)
        weights = weights/(np.sum(weights)+eps)
        idxs = [feat_names.index(n) for n in sel if n in feat_names]
        fr, fh = feats_ref[0, idxs], feats_hat[0, idxs]
        diffs_rel = np.abs(fr - fh)/(np.abs(fr)+eps)
        return 1.0/(1.0+np.sum(weights*diffs_rel))

    def SE_puro(y_orig, y_hat, win=11, blocks=4, eps=1e-12):
        def smooth(x, w): return np.convolve(x, np.ones(w)/w, mode="same") if w>1 else x
        def norm(x): return (x-np.mean(x))/(np.std(x)+eps)
        yo, yh = smooth(y_orig, win), smooth(y_hat, win)
        yo, yh = norm(yo), norm(yh)
        num = np.dot(yo,yh); den = np.linalg.norm(yo)*np.linalg.norm(yh)+eps
        global_corr = (num/den+1)/2.0
        n=len(yo); step=n//blocks; local_corrs=[]
        for b in range(blocks):
            s=(b+1)*step if b<blocks-1 else n
            r1, r2 = yo[b*step:s], yh[b*step:s]
            num=np.dot(r1,r2); den=np.linalg.norm(r1)*np.linalg.norm(r2)+eps
            local_corrs.append((num/den+1)/2.0)
        return 0.5*global_corr+0.5*np.mean(local_corrs)

    def _rmsd(y_ref, y): 
        return float(np.sqrt(np.mean((y_ref - y)**2)))
    def _ccdm(y_ref, y):
        y1, y2 = y_ref - y_ref.mean(), y - y.mean()
        den = (np.linalg.norm(y1)*np.linalg.norm(y2)) + 1e-12
        rho = float(np.clip(np.dot(y1, y2)/den, -1, 1))
        return 1.0 - rho

    feats_orig, _ = compute_features(X_te, fhz)
    feats_rf, _   = compute_features(Y_rf, fhz)
    feats_pk, _   = compute_features(Y_pk, fhz)
    damage_feats = [n for n in feat_names if n in ["amp","centroid","skew","kurt"]]
    idxs = [feat_names.index(n) for n in damage_feats if n in feat_names]
    def _fdi(Fo,Fc):
        if not idxs: return np.ones(Fo.shape[0])
        diffs = np.abs(Fo[:,idxs] - Fc[:,idxs])
        norm_diffs = diffs/(np.std(Fo[:,idxs],axis=0,keepdims=True)+1e-12)
        return 1.0/(1.0+np.mean(norm_diffs,axis=1))

    st_rf,se_rf,st_pk,se_pk=[],[],[],[]
    for i in range(X_te.shape[0]):
        st_rf.append(ST_puro(y_ref, Y_rf[i], fhz, rf_temp, feat_names))
        se_rf.append(SE_puro(X_te[i], Y_rf[i]))
        st_pk.append(ST_puro(y_ref, Y_pk[i], fhz, rf_temp, feat_names))
        se_pk.append(SE_puro(X_te[i], Y_pk[i]))
    st_rf,se_rf,st_pk,se_pk=map(np.array,[st_rf,se_rf,st_pk,se_pk])

    rmsd_rf = np.array([_rmsd(y_ref, y) for y in Y_rf])
    rmsd_pk = np.array([_rmsd(y_ref, y) for y in Y_pk])
    ccdm_rf = np.array([_ccdm(y_ref, y) for y in Y_rf])
    ccdm_pk = np.array([_ccdm(y_ref, y) for y in Y_pk])

    fdi_rf = _fdi(feats_orig,feats_rf)
    fdi_pk = _fdi(feats_orig,feats_pk)

    stse_rf = 0.5*(st_rf+se_rf)
    stse_pk = 0.5*(st_pk+se_pk)

    return dict(
        RF=dict(ST=st_rf,SE=se_rf,RMSD=rmsd_rf,CCDM=ccdm_rf,FDI=fdi_rf,STSE_plus=stse_rf),
        Park=dict(ST=st_pk,SE=se_pk,RMSD=rmsd_pk,CCDM=ccdm_pk,FDI=fdi_pk,STSE_plus=stse_pk)
    )


# ===================== RODANDO TUDO =====================
base_tr = pd.read_pickle(PKL_TREINO)
base_te = pd.read_pickle(PKL_PROVA)

for fmin, fmax in bands:
    print(f"\n\n=== MÉTRICAS | Faixa {fmin}-{fmax} kHz ===")
    freq_cols_tr, _ = get_freq_columns(base_tr, fmin, fmax)
    freq_cols_te, _ = get_freq_columns(base_te, fmin, fmax)
    common_cols = [c for c in freq_cols_tr if c in freq_cols_te]
    fhz = np.array([extract_freq_hz(c) for c in common_cols], float)
    order = np.argsort(fhz)
    common_cols = [common_cols[i] for i in order]; fhz = fhz[order]

    tr_restr = base_tr[base_tr["temp_c"].isin(TEMPS_TREINO)].copy()
    te_restr = base_te[base_te["temp_c"].isin(TEMPS_PROVA)].copy()
    X_tr = tr_restr[common_cols].to_numpy(float); T_tr = tr_restr["temp_c"].to_numpy(float)
    X_te = te_restr[common_cols].to_numpy(float); T_te = te_restr["temp_c"].to_numpy(float)

    pool_20=[]
    if (base_tr["temp_c"]==REF_TEMP).any():
        pool_20.append(base_tr.loc[base_tr["temp_c"]==REF_TEMP, common_cols].to_numpy(float))
    if (base_te["temp_c"]==REF_TEMP).any():
        pool_20.append(base_te.loc[base_te["temp_c"]==REF_TEMP, common_cols].to_numpy(float))
    y_ref = np.median(np.vstack(pool_20), axis=0)

    F_tr, feat_names = compute_features(X_tr, fhz)
    rf_temp_restr = RandomForestRegressor(n_estimators=1500, max_depth=30, n_jobs=-1, random_state=42).fit(F_tr, T_tr)
    feat_models = fit_feature_vs_temp_models(F_tr, T_tr, feat_names)
    caps = dict(gain_frac=CAP_GAIN_FRAC, offset_frac=CAP_OFFSET_FRAC, tilt_frac=CAP_TILT_FRAC)
    Y_te_hat, _ = compensate_set_by_features(X_te, fhz, feat_models, REF_TEMP, y_ref, caps)
    Y_te_park, _, _ = park_batch(X_te, y_ref, fhz)

    for temp_eval in [46, 70]:
        mask = (T_te == temp_eval)
        if np.any(mask):
            print(f"\n--- Temperatura {temp_eval} °C ---")
            res = run_all_metrics(y_ref, X_te[mask], Y_te_hat[mask], Y_te_park[mask], fhz, rf_temp_restr, feat_names)
            for metodo in ["RF","Park"]:
                vals = res[metodo]
                print(f"{metodo}: "
                      f"ST={np.mean(vals['ST']):.3f}, "
                      f"SE={np.mean(vals['SE']):.3f}, "
                      f"RMSD={np.mean(vals['RMSD']):.3f}, "
                      f"CCDM={np.mean(vals['CCDM']):.3f}, "
                      f"FDI={np.mean(vals['FDI']):.3f}, "
                      f"STSE+={np.mean(vals['STSE_plus']):.3f}")
